In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import json 
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from shapely.geometry import box
import xarray as xr

# Simplify GeoJSONS

In [5]:
from pathlib import Path
from IPython.display import display

# Find the dashboard root even if the notebook is opened from legacy_files/
project_root = Path.cwd()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "assets" / "data").exists():
        project_root = candidate
        break

data_root = project_root / "assets" / "data"

startup_loaded = {
    "addis/mpi/addis_districts_MPI.geojson": "Startup: Addis MPI",
    "addis/food_environment/addis_diet_env_mapping.geojson": "Startup: Addis food environment",
    "hanoi/mpi/hanoi_districts_MPI.geojson": "Startup: Hanoi MPI",
    "hanoi/food_environment/hanoi_diet_env_mapping.geojson": "Startup: Hanoi food environment",
    "hanoi/resilience/lulc_stats_gdf.geojson": "Startup: Hanoi LULC",
    "hanoi/resilience/precomputed_hanoi_climate_vars/resilience_districts_base.geojson": "Startup: Hanoi drought districts",
    "hanoi/resilience/precomputed_hanoi_climate_vars/vnm_regions.geojson": "Startup: Hanoi regions",
}

rows = []
for path in sorted(data_root.rglob("*.geojson")):
    rel = path.relative_to(data_root).as_posix()
    size_mb = path.stat().st_size / (1024 * 1024)

    if rel in startup_loaded:
        usage = startup_loaded[rel]
        weight = 3.0
    elif "isochrones_" in rel:
        usage = "On demand: food-environment isochrones"
        weight = 2.0
    elif "jsons_" in rel:
        usage = "On demand: food outlet points"
        weight = 1.5
    else:
        usage = "Other"
        weight = 1.0

    rows.append({
        "relative_path": rel,
        "size_mb": round(size_mb, 3),
        "usage": usage,
        "priority_score": round(size_mb * weight, 3),
    })

geojson_inventory = pd.DataFrame(rows).sort_values(["priority_score", "size_mb"], ascending=False).reset_index(drop=True)

print(f"Found {len(geojson_inventory)} GeoJSON files under {data_root}")
print("\nGeoJSON totals by usage class:\n")
display(
    geojson_inventory.groupby("usage", as_index=False)
    .agg(file_count=("relative_path", "count"), total_mb=("size_mb", "sum"))
    .sort_values("total_mb", ascending=False)
)

print("\nTop priority reduction targets:\n")
display(geojson_inventory.head(20))

priority_targets = geojson_inventory[
    (geojson_inventory["size_mb"] >= 0.2)
    | (geojson_inventory["usage"].str.contains("Startup"))
].copy()

print("\nRecommended first-pass shortlist:\n")
display(priority_targets[["relative_path", "size_mb", "usage"]].head(12))

Found 243 GeoJSON files under /Users/jemimaofarrell/Documents/Python/EcoFoodSystems/EcoFoodSystems_Dashboard_Development/assets/data

GeoJSON totals by usage class:



,usage,file_count,total_mb
2,Other,12,21.706
1,On demand: food-environment isochrones,112,14.030
7,Startup: Hanoi drought districts,1,4.434
0,On demand: food outlet points,112,2.712
8,Startup: Hanoi food environment,1,2.091
9,Startup: Hanoi regions,1,1.085
5,Startup: Hanoi LULC,1,0.792
6,Startup: Hanoi MPI,1,0.788
4,Startup: Addis food environment,1,0.025
3,Startup: Addis MPI,1,0.019



Top priority reduction targets:



,relative_path,size_mb,usage,priority_score
0,hanoi/resilience/precomputed_hanoi_climate_var...,4.434,Startup: Hanoi drought districts,13.301
1,hanoi/food_environment/hanoi_diet_env_mapping....,2.091,Startup: Hanoi food environment,6.274
2,legacy_geojson_backups/hanoi/resilience/precom...,5.030,Other,5.030
3,hanoi/resilience/precomputed_hanoi_climate_var...,4.309,Other,4.309
4,legacy_geojson_backups/hanoi/resilience/precom...,4.309,Other,4.309
5,hanoi/food_environment/isochrones_hanoi/amenit...,1.963,On demand: food-environment isochrones,3.926
6,hanoi/resilience/precomputed_hanoi_climate_var...,1.085,Startup: Hanoi regions,3.256
7,legacy_geojson_backups/hanoi/food_environment/...,1.570,On demand: food-environment isochrones,3.140
8,hanoi/food_environment/isochrones_hanoi/amenit...,1.479,On demand: food-environment isochrones,2.959
9,hanoi/food_environment/legacy/hanoi_diet_env_m...,2.884,Other,2.884



Recommended first-pass shortlist:



,relative_path,size_mb,usage
0,hanoi/resilience/precomputed_hanoi_climate_var...,4.434,Startup: Hanoi drought districts
1,hanoi/food_environment/hanoi_diet_env_mapping....,2.091,Startup: Hanoi food environment
2,legacy_geojson_backups/hanoi/resilience/precom...,5.030,Other
3,hanoi/resilience/precomputed_hanoi_climate_var...,4.309,Other
4,legacy_geojson_backups/hanoi/resilience/precom...,4.309,Other
5,hanoi/food_environment/isochrones_hanoi/amenit...,1.963,On demand: food-environment isochrones
6,hanoi/resilience/precomputed_hanoi_climate_var...,1.085,Startup: Hanoi regions
7,legacy_geojson_backups/hanoi/food_environment/...,1.570,On demand: food-environment isochrones
8,hanoi/food_environment/isochrones_hanoi/amenit...,1.479,On demand: food-environment isochrones
9,hanoi/food_environment/legacy/hanoi_diet_env_m...,2.884,Other


In [9]:
# Test reduction for the largest startup GeoJSON: Hanoi drought districts
from pathlib import Path

# Reuse paths from the previous cell when available
try:
    data_root
    project_root
except NameError:
    project_root = Path.cwd()
    for candidate in [project_root, *project_root.parents]:
        if (candidate / "assets" / "data").exists():
            project_root = candidate
            break
    data_root = project_root / "assets" / "data"

target_rel = "hanoi/resilience/precomputed_hanoi_climate_vars/resilience_districts_base.geojson"
target_path = data_root / target_rel
output_dir = project_root / "legacy_files" / "reduced_geojson_tests"
output_dir.mkdir(parents=True, exist_ok=True)


def count_vertices(geom):
    if geom is None or geom.is_empty:
        return 0
    if geom.geom_type == "Polygon":
        total = len(geom.exterior.coords)
        total += sum(len(ring.coords) for ring in geom.interiors)
        return total
    if geom.geom_type == "MultiPolygon":
        return sum(count_vertices(part) for part in geom.geoms)
    return 0


gdf_original = gpd.read_file(target_path).to_crs("EPSG:4326")
metric_crs = gdf_original.estimate_utm_crs() or "EPSG:32648"
gdf_metric = gdf_original.to_crs(metric_crs)

original_size_mb = target_path.stat().st_size / (1024 * 1024)
original_area = float(gdf_metric.geometry.area.sum())
original_vertices = int(gdf_metric.geometry.apply(count_vertices).sum())

print(f"Testing: {target_rel}")
print(f"Original size: {original_size_mb:.3f} MB | Features: {len(gdf_original)} | Vertices: {original_vertices:,} | Metric CRS: {metric_crs}")

tolerances_m = [10, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500]
results = []

for tol in tolerances_m:
    try:
        simplified_metric = gdf_metric.copy()
        simplified_metric["geometry"] = simplified_metric.geometry.buffer(0).simplify(tol, preserve_topology=True)
        simplified_metric = simplified_metric[simplified_metric.geometry.notna() & ~simplified_metric.geometry.is_empty].copy()
        simplified_metric["geometry"] = simplified_metric.geometry.buffer(0)

        simplified_wgs84 = simplified_metric.to_crs("EPSG:4326")
        out_path = output_dir / f"resilience_districts_base_simplified_{tol}m.geojson"
        simplified_wgs84.to_file(out_path, driver="GeoJSON")

        new_size_mb = out_path.stat().st_size / (1024 * 1024)
        new_area = float(simplified_metric.geometry.area.sum())
        new_vertices = int(simplified_metric.geometry.apply(count_vertices).sum())
        invalid_count = int((~simplified_wgs84.geometry.is_valid).sum())

        results.append({
            "tolerance_m": tol,
            "size_mb": round(new_size_mb, 3),
            "reduction_pct": round((1 - new_size_mb / original_size_mb) * 100, 1),
            "vertices": new_vertices,
            "vertex_reduction_pct": round((1 - new_vertices / original_vertices) * 100, 1),
            "area_change_pct": round(abs(new_area - original_area) / original_area * 100, 3),
            "invalid_geometries": invalid_count,
            "output_path": str(out_path.relative_to(project_root)),
        })
    except Exception as e:
        print(f"Error processing tolerance {tol}m: {e}")

results_df = pd.DataFrame(results).sort_values(["area_change_pct", "size_mb"]).reset_index(drop=True)
display(results_df)

recommended = results_df[(results_df["area_change_pct"] <= 1.0) & (results_df["invalid_geometries"] == 0)]
if recommended.empty:
    recommended = results_df[(results_df["area_change_pct"] <= 2.0) & (results_df["invalid_geometries"] == 0)]

if not recommended.empty:
    best = recommended.sort_values("size_mb").iloc[0]
    print(
        f"\nRecommended test file: {best['output_path']} "
        f"({best['size_mb']:.3f} MB, {best['reduction_pct']:.1f}% smaller, area change {best['area_change_pct']:.3f}%)"
    )
else:
    print("\nNo tolerance met the <=2% area-change rule; inspect the table above and choose the smallest acceptable option manually.")

Testing: hanoi/resilience/precomputed_hanoi_climate_vars/resilience_districts_base.geojson
Original size: 5.030 MB | Features: 708 | Vertices: 114,175 | Metric CRS: EPSG:32649
Error processing tolerance 200m: IllegalArgumentException: CGAlgorithmsDD::orientationIndex encountered NaN/Inf numbers
Error processing tolerance 250m: IllegalArgumentException: CGAlgorithmsDD::orientationIndex encountered NaN/Inf numbers
Error processing tolerance 300m: IllegalArgumentException: CGAlgorithmsDD::orientationIndex encountered NaN/Inf numbers
Error processing tolerance 400m: IllegalArgumentException: CGAlgorithmsDD::orientationIndex encountered NaN/Inf numbers
Error processing tolerance 500m: IllegalArgumentException: CGAlgorithmsDD::orientationIndex encountered NaN/Inf numbers


,tolerance_m,size_mb,reduction_pct,vertices,vertex_reduction_pct,area_change_pct,invalid_geometries,output_path
0,25,5.017,0.2,113911,0.2,0.000,0,legacy_files/reduced_geojson_tests/resilience_...
1,10,5.023,0.1,114058,0.1,0.000,0,legacy_files/reduced_geojson_tests/resilience_...
2,75,4.995,0.7,113397,0.7,0.001,0,legacy_files/reduced_geojson_tests/resilience_...
3,50,5.006,0.5,113666,0.4,0.001,0,legacy_files/reduced_geojson_tests/resilience_...
4,100,4.972,1.1,112865,1.1,0.003,0,legacy_files/reduced_geojson_tests/resilience_...
5,150,3.819,24.1,85826,24.8,0.005,0,legacy_files/reduced_geojson_tests/resilience_...



Recommended test file: legacy_files/reduced_geojson_tests/resilience_districts_base_simplified_150m.geojson (3.819 MB, 24.1% smaller, area change 0.005%)


In [5]:
ADM2 = gpd.read_file("/Users/jemimaofarrell/Documents/Python/EcoFoodSystems/EcoFoodSystems_Dashboard_Development/assets/data/hanoi/resilience/precomputed_hanoi_climate_vars/ADM2_simplified.geojson")

# Cache heavy maps and figures

In [11]:
# Safer topology-preserving reduction tests for district coverages
import json
import shapely


def save_minified_geojson(gdf, path):
    obj = json.loads(gdf.to_json(drop_id=True))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, separators=(",", ":"))


keep_cols = [c for c in ["shapeID", "shapeName", "geometry"] if c in gdf_original.columns]
base = gpd.read_file(target_path)[keep_cols].to_crs(metric_crs).copy()

print("Shapely version:", shapely.__version__)
print("Keeping columns:", keep_cols)

safe_rows = []
set_precision_func = getattr(shapely, "set_precision", None)
coverage_simplify_func = getattr(shapely, "coverage_simplify", None)
coverage_valid_func = getattr(shapely, "coverage_is_valid", None)

if set_precision_func is not None:
    for grid in [1, 5, 10, 20, 50, 100, 150, 200]:
        test = base.copy()
        test["geometry"] = gpd.GeoSeries(
            [set_precision_func(geom, grid_size=grid) if geom is not None else None for geom in test.geometry],
            index=test.index,
            crs=metric_crs,
        )
        test = test[test.geometry.notna() & ~test.geometry.is_empty].copy()
        test["geometry"] = test.geometry.buffer(0)

        out_path = output_dir / f"resilience_districts_base_precision_{grid}m_min.geojson"
        test_wgs84 = test.to_crs("EPSG:4326")
        save_minified_geojson(test_wgs84, out_path)

        size_mb = out_path.stat().st_size / (1024 * 1024)
        area_change_pct = abs(float(test.geometry.area.sum()) - original_area) / original_area * 100
        coverage_ok = coverage_valid_func(test.geometry.array) if coverage_valid_func is not None else "n/a"

        safe_rows.append({
            "method": f"precision_snap_{grid}m",
            "size_mb": round(size_mb, 3),
            "reduction_pct": round((1 - size_mb / original_size_mb) * 100, 1),
            "area_change_pct": round(area_change_pct, 4),
            "invalid_geometries": int((~test_wgs84.geometry.is_valid).sum()),
            "coverage_valid": coverage_ok,
            "output_path": str(out_path.relative_to(project_root)),
        })
else:
    print("set_precision is not available in this Shapely version.")

if coverage_simplify_func is not None:
    for tol in [50, 100, 150, 200]:
        test = base.copy()
        test["geometry"] = gpd.GeoSeries(
            coverage_simplify_func(test.geometry.array, tolerance=tol),
            index=test.index,
            crs=metric_crs,
        )
        test = test[test.geometry.notna() & ~test.geometry.is_empty].copy()
        test["geometry"] = test.geometry.buffer(0)

        out_path = output_dir / f"resilience_districts_base_coverage_{tol}m_min.geojson"
        test_wgs84 = test.to_crs("EPSG:4326")
        save_minified_geojson(test_wgs84, out_path)

        size_mb = out_path.stat().st_size / (1024 * 1024)
        area_change_pct = abs(float(test.geometry.area.sum()) - original_area) / original_area * 100
        coverage_ok = coverage_valid_func(test.geometry.array) if coverage_valid_func is not None else "n/a"

        safe_rows.append({
            "method": f"coverage_simplify_{tol}m",
            "size_mb": round(size_mb, 3),
            "reduction_pct": round((1 - size_mb / original_size_mb) * 100, 1),
            "area_change_pct": round(area_change_pct, 4),
            "invalid_geometries": int((~test_wgs84.geometry.is_valid).sum()),
            "coverage_valid": coverage_ok,
            "output_path": str(out_path.relative_to(project_root)),
        })
else:
    print("coverage_simplify is not available in this Shapely version.")

safe_df = pd.DataFrame(safe_rows)
if not safe_df.empty:
    safe_df = safe_df.sort_values(["size_mb", "area_change_pct"]).reset_index(drop=True)

display(safe_df)

if not safe_df.empty:
    good = safe_df[(safe_df["invalid_geometries"] == 0) & (safe_df["coverage_valid"].astype(str) != "False")]
    if not good.empty:
        best_safe = good.sort_values(["size_mb", "area_change_pct"]).iloc[0]
        print(
            f"\nBest topology-safe option: {best_safe['method']} -> {best_safe['size_mb']:.3f} MB "
            f"({best_safe['reduction_pct']:.1f}% smaller)"
        )
    else:
        print("\nNo clearly topology-safe reduction outperformed the original enough to recommend aggressive simplification.")

Shapely version: 2.0.4
Keeping columns: ['shapeID', 'shapeName', 'geometry']
coverage_simplify is not available in this Shapely version.


,method,size_mb,reduction_pct,area_change_pct,invalid_geometries,coverage_valid,output_path
0,precision_snap_200m,4.309,14.3,0.0004,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
1,precision_snap_150m,4.368,13.2,0.0010,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
2,precision_snap_100m,4.397,12.6,0.0001,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
3,precision_snap_1m,4.398,12.5,0.0000,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
4,precision_snap_5m,4.398,12.6,0.0000,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
5,precision_snap_10m,4.399,12.5,0.0000,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
6,precision_snap_20m,4.399,12.5,0.0001,0,n/a,legacy_files/reduced_geojson_tests/resilience_...
7,precision_snap_50m,4.400,12.5,0.0008,0,n/a,legacy_files/reduced_geojson_tests/resilience_...



Best topology-safe option: precision_snap_200m -> 4.309 MB (14.3% smaller)


In [12]:
# Test reduction for Startup: Hanoi food environment and compare storage patterns
food_rel = "hanoi/food_environment/hanoi_diet_env_mapping.geojson"
food_path = data_root / food_rel

food_gdf = gpd.read_file(food_path)
food_original_size_mb = food_path.stat().st_size / (1024 * 1024)
food_metric_crs = food_gdf.estimate_utm_crs() or "EPSG:32648"
food_metric = food_gdf.to_crs(food_metric_crs).copy()

# Keep likely join/id columns and all numeric indicators for comparison
id_candidates = ["shapeID", "Dist_Name", "Dist_name", "district", "name"]
keep_id_cols = [c for c in id_candidates if c in food_gdf.columns]
if not keep_id_cols and len(food_gdf.columns) > 1:
    keep_id_cols = [c for c in food_gdf.columns if c != "geometry"][:1]

numeric_cols = [c for c in food_gdf.columns if c != "geometry" and pd.api.types.is_numeric_dtype(food_gdf[c])]
base_cols = list(dict.fromkeys(keep_id_cols + ["geometry"]))
attr_cols = list(dict.fromkeys(keep_id_cols + numeric_cols))

print(f"Testing: {food_rel}")
print(f"Original size: {food_original_size_mb:.3f} MB | Features: {len(food_gdf)} | Columns: {len(food_gdf.columns) - 1} attrs")
print("Join/id columns:", keep_id_cols)
print(f"Numeric indicator columns: {len(numeric_cols)}")

# 1) Safer reduction: precision snap + minified GeoJSON with all columns retained
food_snap = food_metric.copy()
food_snap["geometry"] = gpd.GeoSeries(
    [shapely.set_precision(geom, grid_size=1) if geom is not None else None for geom in food_snap.geometry],
    index=food_snap.index,
    crs=food_metric_crs,
)
food_snap = food_snap[food_snap.geometry.notna() & ~food_snap.geometry.is_empty].copy()
food_snap["geometry"] = food_snap.geometry.buffer(0)
food_snap_wgs84 = food_snap.to_crs("EPSG:4326")

food_min_path = output_dir / "hanoi_diet_env_mapping_precision_1m_min.geojson"
save_minified_geojson(food_snap_wgs84, food_min_path)
food_min_size_mb = food_min_path.stat().st_size / (1024 * 1024)

# 2) Split pattern: base geometry GeoJSON + CSV of attributes
food_base = food_snap_wgs84[base_cols].copy()
food_base_path = output_dir / "hanoi_diet_env_mapping_base_precision_1m_min.geojson"
save_minified_geojson(food_base, food_base_path)
food_base_size_mb = food_base_path.stat().st_size / (1024 * 1024)

food_attrs = food_gdf[attr_cols].copy()
food_csv_path = output_dir / "hanoi_diet_env_mapping_attributes.csv"
food_attrs.to_csv(food_csv_path, index=False)
food_csv_size_mb = food_csv_path.stat().st_size / (1024 * 1024)

comparison_df = pd.DataFrame([
    {
        "option": "Original full GeoJSON",
        "size_mb": round(food_original_size_mb, 3),
        "savings_pct": 0.0,
        "notes": "Current file with geometry + all properties"
    },
    {
        "option": "Minified full GeoJSON",
        "size_mb": round(food_min_size_mb, 3),
        "savings_pct": round((1 - food_min_size_mb / food_original_size_mb) * 100, 1),
        "notes": "Same structure, safer precision snap + compact JSON"
    },
    {
        "option": "Base GeoJSON + CSV",
        "size_mb": round(food_base_size_mb + food_csv_size_mb, 3),
        "savings_pct": round((1 - (food_base_size_mb + food_csv_size_mb) / food_original_size_mb) * 100, 1),
        "notes": f"{food_base_size_mb:.3f} MB base geometry + {food_csv_size_mb:.3f} MB CSV"
    },
])

display(comparison_df)

print("\nBase geometry columns:", base_cols)
print("Attribute file columns (first 12):", attr_cols[:12])
print(f"\nOutputs written to: {output_dir.relative_to(project_root)}")

Testing: hanoi/food_environment/hanoi_diet_env_mapping.geojson
Original size: 2.884 MB | Features: 126 | Columns: 20 attrs
Join/id columns: ['ma_xa']
Numeric indicator columns: 12


,option,size_mb,savings_pct,notes
0,Original full GeoJSON,2.884,0.0,Current file with geometry + all properties
1,Minified full GeoJSON,2.172,24.7,"Same structure, safer precision snap + compact..."
2,Base GeoJSON + CSV,2.099,27.2,2.091 MB base geometry + 0.008 MB CSV



Base geometry columns: ['ma_xa', 'geometry']
Attribute file columns (first 12): ['ma_xa', 'cap', 'stt', 'dtich_km2', 'dan_so', 'matdo_km2', 'Cnt_Count_healthy', 'density_healthyout', 'cnt_Count_Unhealthy', 'density_unhealthyout', 'Count_Mixoutlets', 'density_mixoutlets']

Outputs written to: legacy_files/reduced_geojson_tests


In [6]:
# Bulk-apply the safe slimming approach to all dashboard GeoJSONs (excluding legacy backups)
import shutil

bulk_backup_root = data_root / "_geojson_legacy_backups_20260401"
bulk_backup_root.mkdir(parents=True, exist_ok=True)
bulk_report_path = project_root / "legacy_files" / "reduced_geojson_tests" / "geojson_bulk_optimization_report.csv"

bulk_rows = []
total_before_bytes = 0
total_after_bytes = 0

geojson_paths = [
    p for p in sorted(data_root.rglob("*.geojson"))
    if "legacy" not in p.parts and "_geojson_legacy_backups_20260401" not in p.parts
]

for path in geojson_paths:
    rel = path.relative_to(data_root)
    before_bytes = path.stat().st_size
    total_before_bytes += before_bytes

    backup_path = bulk_backup_root / rel
    backup_path.parent.mkdir(parents=True, exist_ok=True)
    if not backup_path.exists():
        shutil.copy2(path, backup_path)

    try:
        gdf = gpd.read_file(path)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326", allow_override=True)

        working = gdf.copy()
        geom_types = set(working.geom_type.dropna().astype(str).unique()) if "geometry" in working else set()

        if hasattr(shapely, "set_precision") and not working.empty and geom_types:
            try:
                metric_crs_local = working.estimate_utm_crs() if working.crs and working.crs.is_geographic else working.crs
            except Exception:
                metric_crs_local = None

            if metric_crs_local:
                working = working.to_crs(metric_crs_local)
                working["geometry"] = gpd.GeoSeries(
                    [shapely.set_precision(geom, grid_size=1) if geom is not None else None for geom in working.geometry],
                    index=working.index,
                    crs=metric_crs_local,
                )
                if any(gt in {"Polygon", "MultiPolygon"} for gt in geom_types):
                    working = working[working.geometry.notna() & ~working.geometry.is_empty].copy()
                    working["geometry"] = working.geometry.buffer(0)
                working = working.to_crs("EPSG:4326")
            elif working.crs and str(working.crs).upper() != "EPSG:4326":
                working = working.to_crs("EPSG:4326")
        elif working.crs and str(working.crs).upper() != "EPSG:4326":
            working = working.to_crs("EPSG:4326")

        save_minified_geojson(working, path)
        after_bytes = path.stat().st_size
        total_after_bytes += after_bytes

        bulk_rows.append({
            "relative_path": rel.as_posix(),
            "before_mb": round(before_bytes / (1024 * 1024), 3),
            "after_mb": round(after_bytes / (1024 * 1024), 3),
            "saved_mb": round((before_bytes - after_bytes) / (1024 * 1024), 3),
            "saved_pct": round((1 - after_bytes / before_bytes) * 100, 1) if before_bytes else 0.0,
            "status": "optimized",
        })
    except Exception as exc:
        total_after_bytes += before_bytes
        bulk_rows.append({
            "relative_path": rel.as_posix(),
            "before_mb": round(before_bytes / (1024 * 1024), 3),
            "after_mb": round(before_bytes / (1024 * 1024), 3),
            "saved_mb": 0.0,
            "saved_pct": 0.0,
            "status": f"error: {exc}",
        })

bulk_report_df = pd.DataFrame(bulk_rows).sort_values(["saved_mb", "before_mb"], ascending=False).reset_index(drop=True)
bulk_report_df.to_csv(bulk_report_path, index=False)

print(f"Processed {len(bulk_report_df)} GeoJSON files")
print(f"Total before: {total_before_bytes / (1024 * 1024):.3f} MB")
print(f"Total after:  {total_after_bytes / (1024 * 1024):.3f} MB")
print(f"Total saved:  {(total_before_bytes - total_after_bytes) / (1024 * 1024):.3f} MB ({(1 - total_after_bytes / total_before_bytes) * 100:.1f}%)")
print(f"Backups saved under: {bulk_backup_root.relative_to(project_root)}")
print(f"Report saved to: {bulk_report_path.relative_to(project_root)}")

print("\nTop 20 savings:\n")
display(bulk_report_df.head(20))

Processed 242 GeoJSON files
Total before: 44.801 MB
Total after:  44.801 MB
Total saved:  0.000 MB (0.0%)
Backups saved under: assets/data/_geojson_legacy_backups_20260401
Report saved to: legacy_files/reduced_geojson_tests/geojson_bulk_optimization_report.csv

Top 20 savings:



,relative_path,before_mb,after_mb,saved_mb,saved_pct,status
0,legacy_geojson_backups/hanoi/resilience/precom...,5.030,5.030,0.0,0.0,error: name 'shapely' is not defined
1,hanoi/resilience/precomputed_hanoi_climate_var...,4.434,4.434,0.0,0.0,error: name 'shapely' is not defined
2,hanoi/resilience/precomputed_hanoi_climate_var...,4.309,4.309,0.0,0.0,error: name 'shapely' is not defined
3,legacy_geojson_backups/hanoi/resilience/precom...,4.309,4.309,0.0,0.0,error: name 'shapely' is not defined
4,hanoi/food_environment/hanoi_diet_env_mapping....,2.091,2.091,0.0,0.0,error: name 'shapely' is not defined
5,legacy_geojson_backups/hanoi/food_environment/...,2.091,2.091,0.0,0.0,error: name 'shapely' is not defined
6,hanoi/food_environment/isochrones_hanoi/amenit...,1.963,1.963,0.0,0.0,error: name 'shapely' is not defined
7,legacy_geojson_backups/hanoi/food_environment/...,1.570,1.570,0.0,0.0,error: name 'shapely' is not defined
8,hanoi/food_environment/isochrones_hanoi/amenit...,1.479,1.479,0.0,0.0,error: name 'shapely' is not defined
9,legacy_geojson_backups/hanoi/resilience/precom...,1.232,1.232,0.0,0.0,error: name 'shapely' is not defined


In [7]:
# Keep only the GeoJSON optimizations that genuinely reduce file size
restore_rows = bulk_report_df[bulk_report_df["after_mb"] >= bulk_report_df["before_mb"]].copy()

for rel in restore_rows["relative_path"]:
    src = bulk_backup_root / rel
    dst = data_root / rel
    if src.exists():
        shutil.copy2(src, dst)

final_rows = []
for rel in bulk_report_df["relative_path"]:
    dst = data_root / rel
    src = bulk_backup_root / rel
    before_bytes = src.stat().st_size if src.exists() else dst.stat().st_size
    after_bytes = dst.stat().st_size
    final_rows.append({
        "relative_path": rel,
        "before_mb": round(before_bytes / (1024 * 1024), 3),
        "after_mb": round(after_bytes / (1024 * 1024), 3),
        "saved_mb": round((before_bytes - after_bytes) / (1024 * 1024), 3),
        "saved_pct": round((1 - after_bytes / before_bytes) * 100, 1) if before_bytes else 0.0,
        "status": "kept" if after_bytes < before_bytes else "restored",
    })

final_report_df = pd.DataFrame(final_rows).sort_values(["saved_mb", "before_mb"], ascending=False).reset_index(drop=True)
final_report_df.to_csv(bulk_report_path, index=False)

final_before = final_report_df["before_mb"].sum()
final_after = final_report_df["after_mb"].sum()
restored_count = int((final_report_df["status"] == "restored").sum())
kept_count = int((final_report_df["status"] == "kept").sum())

print(f"Restored {restored_count} files that did not shrink.")
print(f"Kept optimized versions for {kept_count} files.")
print(f"Final before: {final_before:.3f} MB")
print(f"Final after:  {final_after:.3f} MB")
print(f"Final saved:  {final_before - final_after:.3f} MB ({(1 - final_after / final_before) * 100:.1f}%)")

print("\nTop retained savings:\n")
display(final_report_df[final_report_df["saved_mb"] > 0].head(20))

Restored 242 files that did not shrink.
Kept optimized versions for 0 files.
Final before: 44.798 MB
Final after:  44.798 MB
Final saved:  0.000 MB (0.0%)

Top retained savings:



,relative_path,before_mb,after_mb,saved_mb,saved_pct,status


In [ ]:
# Selective bulk optimization: only keep GeoJSON rewrites that actually reduce file size
import shapely


def to_minified_geojson_text(gdf):
    obj = json.loads(gdf.to_json(drop_id=True))
    return json.dumps(obj, separators=(",", ":"), ensure_ascii=False)


selective_rows = []
sel_before_bytes = 0
sel_after_bytes = 0
kept_count = 0
restored_count = 0

selective_geojson_paths = [
    p for p in sorted(data_root.rglob("*.geojson"))
    if "legacy" not in p.parts
    and "backup" not in " ".join(p.parts).lower()
    and "_geojson_legacy_backups_20260401" not in p.parts
]

for path in selective_geojson_paths:
    rel = path.relative_to(data_root)
    before_bytes = path.stat().st_size
    sel_before_bytes += before_bytes

    try:
        gdf = gpd.read_file(path)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326", allow_override=True)

        working = gdf.copy()
        geom_types = set(working.geom_type.dropna().astype(str).unique()) if "geometry" in working else set()

        if hasattr(shapely, "set_precision") and not working.empty and geom_types:
            try:
                metric_crs_local = working.estimate_utm_crs() if working.crs and working.crs.is_geographic else working.crs
            except Exception:
                metric_crs_local = None

            if metric_crs_local:
                working = working.to_crs(metric_crs_local)
                working["geometry"] = gpd.GeoSeries(
                    [shapely.set_precision(geom, grid_size=1) if geom is not None else None for geom in working.geometry],
                    index=working.index,
                    crs=metric_crs_local,
                )
                if any(gt in {"Polygon", "MultiPolygon"} for gt in geom_types):
                    working = working[working.geometry.notna() & ~working.geometry.is_empty].copy()
                    working["geometry"] = working.geometry.buffer(0)
                working = working.to_crs("EPSG:4326")
            elif working.crs and str(working.crs).upper() != "EPSG:4326":
                working = working.to_crs("EPSG:4326")
        elif working.crs and str(working.crs).upper() != "EPSG:4326":
            working = working.to_crs("EPSG:4326")

        candidate_text = to_minified_geojson_text(working)
        candidate_bytes = len(candidate_text.encode("utf-8"))

        if candidate_bytes < before_bytes:
            with open(path, "w", encoding="utf-8") as f:
                f.write(candidate_text)
            after_bytes = candidate_bytes
            status = "optimized"
            kept_count += 1
        else:
            after_bytes = before_bytes
            status = "unchanged"
            restored_count += 1

        sel_after_bytes += after_bytes
        selective_rows.append({
            "relative_path": rel.as_posix(),
            "before_mb": round(before_bytes / (1024 * 1024), 3),
            "after_mb": round(after_bytes / (1024 * 1024), 3),
            "saved_mb": round((before_bytes - after_bytes) / (1024 * 1024), 3),
            "saved_pct": round((1 - after_bytes / before_bytes) * 100, 1) if before_bytes else 0.0,
            "status": status,
        })
    except Exception as exc:
        sel_after_bytes += before_bytes
        selective_rows.append({
            "relative_path": rel.as_posix(),
            "before_mb": round(before_bytes / (1024 * 1024), 3),
            "after_mb": round(before_bytes / (1024 * 1024), 3),
            "saved_mb": 0.0,
            "saved_pct": 0.0,
            "status": f"error: {exc}",
        })

selective_report_df = pd.DataFrame(selective_rows).sort_values(["saved_mb", "before_mb"], ascending=False).reset_index(drop=True)
selective_report_df.to_csv(bulk_report_path, index=False)

print(f"Kept optimized versions for {kept_count} GeoJSON files.")
print(f"Left {restored_count} files unchanged because they did not shrink.")
print(f"Final before: {sel_before_bytes / (1024 * 1024):.3f} MB")
print(f"Final after:  {sel_after_bytes / (1024 * 1024):.3f} MB")
print(f"Final saved:  {(sel_before_bytes - sel_after_bytes) / (1024 * 1024):.3f} MB ({(1 - sel_after_bytes / sel_before_bytes) * 100:.1f}%)")

print("\nTop retained savings:\n")
display(selective_report_df[selective_report_df["saved_mb"] > 0].head(25))

Kept optimized versions for 49 GeoJSON files.
Left 72 files unchanged because they did not shrink.
Final before: 22.578 MB
Final after:  22.578 MB
Final saved:  0.001 MB (0.0%)

Top retained savings:



,relative_path,before_mb,after_mb,saved_mb,saved_pct,status


: 